In [7]:

import pandas as pd
import numpy as np


In [8]:
data = pd.read_csv("/content/3m.csv")
data['timestamp'] = pd.to_datetime(data['timestamp'])

data = data.set_index('timestamp', inplace=False)
df_3m = data


In [9]:
# prompt: calculate MACD histogram and signal line

# Calculate the 12-day and 26-day exponential moving averages (EMAs)
df_3m['EMA12'] = df_3m['close'].ewm(span=12, adjust=False).mean()
df_3m['EMA26'] = df_3m['close'].ewm(span=26, adjust=False).mean()

# Calculate the MACD line
df_3m['MACD'] = df_3m['EMA12'] - df_3m['EMA26']

# Calculate the 9-day EMA of the MACD line (the signal line)
df_3m['Signal'] = (df_3m['MACD'].ewm(span=9, adjust=False).mean()).rolling(window=20).mean()/(df_3m['MACD'].ewm(span=9, adjust=False).mean()).rolling(window=20).std()

# Calculate the MACD histogram
df_3m['Histogram'] = (df_3m['MACD'] - df_3m['Signal'])

# Display the DataFrame with the MACD, signal line, and histogram
print(df_3m[['MACD', 'Signal', 'Histogram']])


                           MACD    Signal   Histogram
timestamp                                            
2021-06-01 00:00:00    0.000000       NaN         NaN
2021-06-01 00:03:00   13.816524       NaN         NaN
2021-06-01 00:06:00   21.605806       NaN         NaN
2021-06-01 00:09:00   26.731588       NaN         NaN
2021-06-01 00:12:00   25.463507       NaN         NaN
...                         ...       ...         ...
2023-12-29 15:39:00  -60.556618  0.729852  -61.286469
2023-12-29 15:42:00  -72.992049  0.546025  -73.538074
2023-12-29 15:45:00  -92.321324  0.370580  -92.691903
2023-12-29 15:48:00 -112.188746  0.202467 -112.391213
2023-12-29 15:51:00 -135.115202  0.041589 -135.156791

[451998 rows x 3 columns]


In [10]:
# prompt: resample data to df_1h

df_1h = data.resample('1H').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum'})
df_1h


<ipython-input-10-82b9fd2d59c2>:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_1h = data.resample('1H').agg({'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum'})


,open,high,low,close,volume
timestamp,,,,,
2021-06-01 00:00:00,37244.36,37893.76,37100.00,37777.37,28162.765
2021-06-01 01:00:00,37777.36,37787.94,37159.19,37196.69,18117.185
2021-06-01 02:00:00,37201.41,37296.83,36600.00,36848.50,25068.856
2021-06-01 03:00:00,36848.51,37118.71,36458.48,36709.23,22068.260
2021-06-01 04:00:00,36709.23,36950.00,36552.06,36735.97,14661.951
...,...,...,...,...,...
2023-12-29 11:00:00,42726.20,42996.30,42715.00,42908.90,12612.116
2023-12-29 12:00:00,42908.80,42979.20,42789.10,42865.80,8694.228
2023-12-29 13:00:00,42865.80,42926.10,42743.00,42865.00,9438.783


In [11]:
# prompt: calculate ATR of df-1h

import numpy as np
# Calculate True Range (TR)
df_1h['TR'] = 0.0
df_1h['TR'] = np.maximum(df_1h['high'] - df_1h['low'],
                        np.maximum(abs(df_1h['high'] - df_1h['close'].shift()),
                                   abs(df_1h['low'] - df_1h['close'].shift())))


# Calculate Average True Range (ATR)
atr_period = 14  # Define the ATR period (e.g., 14)
df_1h['ATR'] = df_1h['TR'].rolling(window=atr_period).mean()
(df_1h[['TR', 'ATR']])


,TR,ATR
timestamp,,
2021-06-01 00:00:00,NaN,NaN
2021-06-01 01:00:00,628.75,NaN
2021-06-01 02:00:00,696.83,NaN
2021-06-01 03:00:00,660.23,NaN
2021-06-01 04:00:00,397.94,NaN
...,...,...
2023-12-29 11:00:00,281.30,278.300000
2023-12-29 12:00:00,190.10,273.428571
2023-12-29 13:00:00,183.10,270.985714


In [12]:
df_1h = df_1h.resample('3T').ffill(limit=20)
df_1h

<ipython-input-12-5df6a21cef4d>:1: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_1h = df_1h.resample('3T').ffill(limit=20)


,open,high,low,close,volume,TR,ATR
timestamp,,,,,,,
2021-06-01 00:00:00,37244.36,37893.76,37100.0,37777.37,28162.765,NaN,NaN
2021-06-01 00:03:00,37244.36,37893.76,37100.0,37777.37,28162.765,NaN,NaN
2021-06-01 00:06:00,37244.36,37893.76,37100.0,37777.37,28162.765,NaN,NaN
2021-06-01 00:09:00,37244.36,37893.76,37100.0,37777.37,28162.765,NaN,NaN
2021-06-01 00:12:00,37244.36,37893.76,37100.0,37777.37,28162.765,NaN,NaN
...,...,...,...,...,...,...,...
2023-12-29 14:48:00,42865.00,43171.90,42800.0,43065.60,14731.325,371.9,274.985714
2023-12-29 14:51:00,42865.00,43171.90,42800.0,43065.60,14731.325,371.9,274.985714
2023-12-29 14:54:00,42865.00,43171.90,42800.0,43065.60,14731.325,371.9,274.985714


In [13]:
# prompt: join df['ATR'] with df_3m

# Merge df_1h['ATR'] with df_3m based on the timestamp index.
df_merged = df_3m.merge(df_1h['ATR'], left_index=True, right_index=True, how='left')

# Display the merged DataFrame.
final_data = df_merged.dropna()


In [14]:
# prompt: Price multiplier (PM) = (0.5 * MACD.HSTG.Sig + 0.5 * MACD DM Signal) * NATR/2
# Adjusted Mid Price (AMP) = Last price * (1+PM)
#  Order price = AMP * 0.998

# Assuming 'final_data' DataFrame from the previous code is available.

# Calculate NATR (Normalized Average True Range)
final_data['NATR'] = (final_data['ATR'] / final_data['close']) * 100

# Assuming MACD.HSTG.Sig and MACD DM Signal are available in the DataFrame
# Replace these placeholders with your actual column names if different



# Calculate Price Multiplier (PM)
final_data['PM'] = (0.5 * final_data['MACD.HSTG.Sig'] + 0.5 * final_data['MACD_DM_Signal']) * (final_data['NATR'] / 2)

# Calculate Adjusted Mid Price (AMP)
final_data['AMP'] = final_data['close'] * (1 + final_data['PM'])

# Calculate Order Price
final_data['Order_Price'] = final_data['AMP'] * 0.998

final_data


<ipython-input-14-8925ca9ab599>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_data['NATR'] = (final_data['ATR'] / final_data['close']) * 100


KeyError: 'MACD.HSTG.Sig'